# Toxic Comment / Hate Speech Detection
### Detectron AI · Module 09

This notebook trains a **TF-IDF + Linear SVM** classifier on labeled comments to detect
toxicity, and compares it with the weighted-lexicon scorer used in the live web demo —
the same family of technique behind early content-moderation systems before
transformer-based models (like the real Perspective API) took over.


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


## 1. Labeled comment dataset (toxic=1, non-toxic=0)

In [2]:
data = [
    ("You are such an idiot, nobody likes you", 1),
    ("Shut up you stupid moron, you're pathetic", 1),
    ("I hate you, you disgusting piece of garbage", 1),
    ("Just go away, you're worthless trash", 1),
    ("You ugly loser, everyone thinks you're dumb", 1),
    ("Get lost you scum, nobody wants you here", 1),
    ("You're so stupid it's actually disgusting", 1),
    ("What a pathetic moron, complete waste of space", 1),
    ("I can't believe how dumb and worthless you are", 1),
    ("Everyone hates you, you absolute idiot", 1),
    ("Thanks for the feedback, I'll revise the draft tonight", 0),
    ("Great point, I hadn't considered that angle before", 0),
    ("Could you clarify what you meant in the second paragraph?", 0),
    ("I appreciate you taking the time to explain this", 0),
    ("This is a really well-written article, thank you for sharing", 0),
    ("I disagree with this conclusion, here's my reasoning", 0),
    ("Nice work on the presentation, the data was clear", 0),
    ("Let's schedule a call to discuss the next steps", 0),
    ("I think there might be an error in the third row", 0),
    ("Happy to help if you need anything else", 0),
]

df = pd.DataFrame(data, columns=["text", "label"])
df["label_name"] = df["label"].map({1: "toxic", 0: "non-toxic"})
df


,text,label,label_name
0,"You are such an idiot, nobody likes you",1,toxic
1,"Shut up you stupid moron, you're pathetic",1,toxic
2,"I hate you, you disgusting piece of garbage",1,toxic
3,"Just go away, you're worthless trash",1,toxic
4,"You ugly loser, everyone thinks you're dumb",1,toxic
5,"Get lost you scum, nobody wants you here",1,toxic
6,You're so stupid it's actually disgusting,1,toxic
7,"What a pathetic moron, complete waste of space",1,toxic
8,I can't believe how dumb and worthless you are,1,toxic
9,"Everyone hates you, you absolute idiot",1,toxic


## 2. TF-IDF + Linear SVM

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LinearSVC(class_weight="balanced")
model.fit(X_train_vec, y_train)
pred = model.predict(X_test_vec)

print(classification_report(y_test, pred, target_names=["non-toxic", "toxic"]))


              precision    recall  f1-score   support

   non-toxic       1.00      1.00      1.00         3
       toxic       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



## 3. Most predictive words for toxicity

In [4]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_[0]
top_toxic = feature_names[np.argsort(coefs)[-10:][::-1]]
top_safe = feature_names[np.argsort(coefs)[:10]]
print("Top toxic-indicating terms:", top_toxic)
print("Top non-toxic-indicating terms:", top_safe)


Top toxic-indicating terms: ['you' 'you re' 're' 'pathetic' 'moron' 'nobody' 'worthless' 'you are'
 'are' 'dumb']
Top non-toxic-indicating terms: ['the' 'to' 'in' 'in the' 'here my' 'my reasoning' 'conclusion here'
 'conclusion' 'my' 'reasoning']


## 4. Weighted lexicon scorer (matches the live web demo)

In [5]:
TOXIC_LEXICON = {
    "idiot": 25, "stupid": 20, "hate you": 35, "shut up": 20, "worthless": 30,
    "pathetic": 25, "trash": 18, "loser": 20, "ugly": 15, "dumb": 18, "moron": 28,
    "disgusting": 22, "scum": 30, "garbage": 18, "go away": 12, "nobody likes you": 35,
}

def analyze_toxicity(text):
    lower = text.lower()
    score = 0
    hits = []
    for phrase, weight in TOXIC_LEXICON.items():
        if phrase in lower:
            score += weight
            hits.append((phrase, weight))
    caps = sum(1 for w in text.split() if w.isupper() and len(w) >= 3)
    if caps:
        score += caps * 6
        hits.append((f"{caps} all-caps words", caps * 6))
    score = min(100, score)
    verdict = "toxic" if score >= 50 else "mildly offensive" if score >= 20 else "non-toxic"
    return score, verdict, hits

score, verdict, hits = analyze_toxicity("You are such an IDIOT, nobody likes you and you should just go away!!!")
print(f"Score: {score}/100 -> {verdict.upper()}")
for h in hits: print(" -", h)


Score: 78/100 -> TOXIC
 - ('idiot', 25)
 - ('go away', 12)
 - ('nobody likes you', 35)
 - ('1 all-caps words', 6)


## Notes

- TF-IDF + Linear SVM learns toxicity patterns statistically from labeled examples,
  while the weighted lexicon directly encodes known abusive terms — fast and
  interpretable, but brittle to novel phrasing/sarcasm the lexicon doesn't cover.
- Real production systems (like Google's **Perspective API**, the real-world inspiration
  for this module) use large transformer models trained on millions of moderated
  comments, which generalize far better to context, sarcasm, and coded language.
